# Predicting Your Perfect Cup!

Members: Ashly Turcios Sierra and Reagan McGowan

In [ ]:
# Install packages
!pip install -q -r ../requirements.txt

#for modeling
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as smf
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix,
                             classification_report)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
import statsmodels.api as sm

# REPRODUCABILITY
RANDOM_SEED = 30220
np.random.seed(RANDOM_SEED)


### Dataset: Coffee Quality Database

In [ ]:
# Load files
DATA_DIR = "../data"
arabica = pd.read_csv(f"{DATA_DIR}/arabica_ratings_raw.csv")
robusta = pd.read_csv(f"{DATA_DIR}/robusta_ratings_raw.csv")

# Define common renaming map for sensory attributes
rename_map = {
    "Fragrance/Aroma": "Aroma",
    "Salt...Acid": "Acidity",
    "Bitter/Sweet": "Sweetness",
    "Mouthfeel": "Body",
    "Uniform.Cup": "Uniformity",
    "Clean.Cup": "Clean Cup",
    "Cupper.Points": "Cupper Points",
    "Total.Cup.Points": "Total Cup Points"
}

# Rename columns in both datasets to standardize sensory attribute names
arabica = arabica.rename(columns=rename_map)
robusta = robusta.rename(columns=rename_map)

# Keep only common columns after renaming
common_cols = sorted(list(set(arabica.columns).intersection(set(robusta.columns))))
arabica = arabica[common_cols].copy()
robusta = robusta[common_cols].copy()

# Combine
coffee = pd.concat([arabica, robusta], ignore_index=True)

print("Combined shape:", coffee.shape)
display(coffee.head())

Combined shape: (1340, 47)


,Aftertaste,Altitude,Bag Weight,Balance,Body,Category One Defects,Category Two Defects,Certification Address,Certification Body,Certification Contact,...,Request a Sample,Species,Status,Total Cup Points,Unnamed: 0,Variety,View Green Analysis Details,quality_score,view_certificate_1,view_certificate_2
0,8.67,1950-2200,60 kg,8.42,8.50,0 full defects,0 full defects,"BAWA Center, 3rd Floor (Gerji), Addis Ababa, E...",METAD Agricultural Development plc,"Aman Adinew (Emebet Dinku) - +251-116-292534, ...",...,NaN,Arabica,Completed,Sample 90.58,0,NaN,NaN,90.58,NaN,NaN
1,8.50,1950-2200,60 kg,8.42,8.42,0 full defects,1 full defects,"BAWA Center, 3rd Floor (Gerji), Addis Ababa, E...",METAD Agricultural Development plc,"Aman Adinew (Emebet Dinku) - +251-116-292534, ...",...,NaN,Arabica,Completed,Sample 89.92,1,Other,NaN,89.92,NaN,NaN
2,8.42,1600 - 1800 m,1,8.42,8.33,0 full defects,0 full defects,"117 W 4th St, Suite 300 Santa Ana, CA 92701",Specialty Coffee Association,Chris Buck - (562) 624-4100,...,NaN,Arabica,Completed,Sample 89.75,2,Bourbon,NaN,89.75,NaN,NaN
3,8.42,1800-2200,60 kg,8.25,8.50,0 full defects,2 full defects,"BAWA Center, 3rd Floor (Gerji), Addis Ababa, E...",METAD Agricultural Development plc,"Aman Adinew (Emebet Dinku) - +251-116-292534, ...",...,NaN,Arabica,Completed,Sample 89.00,3,NaN,NaN,89.00,NaN,NaN
4,8.25,1950-2200,60 kg,8.33,8.42,0 full defects,2 full defects,"BAWA Center, 3rd Floor (Gerji), Addis Ababa, E...",METAD Agricultural Development plc,"Aman Adinew (Emebet Dinku) - +251-116-292534, ...",...,NaN,Arabica,Completed,Sample 88.83,4,Other,NaN,88.83,NaN,NaN


In [ ]:
#Ensuring the two datasets have the same columns in order to combine into coffee
display(robusta.columns)
display(arabica.columns)
display(coffee.columns)

Index(['Aftertaste', 'Altitude', 'Bag Weight', 'Balance', 'Body',
       'Category One Defects', 'Category Two Defects', 'Certification Address',
       'Certification Body', 'Certification Contact', 'Clean Cup', 'Color',
       'Company', 'Country of Origin', 'Cupper Points',
       'Cupping Protocol and Descriptors', 'Expiration', 'Farm Name', 'Flavor',
       'Grading Date', 'Harvest Year', 'ICO Number', 'In-Country Partner',
       'Lot Number', 'Mill', 'Moisture', 'NA', 'NA.1', 'NA.2', 'NA.3',
       'Number of Bags', 'Owner', 'Owner.1', 'Processing Method', 'Producer',
       'Quakers', 'Region', 'Request a Sample', 'Species', 'Status',
       'Total Cup Points', 'Unnamed: 0', 'Variety',
       'View Green Analysis Details', 'quality_score', 'view_certificate_1',
       'view_certificate_2'],
      dtype='object')

Index(['Aftertaste', 'Altitude', 'Bag Weight', 'Balance', 'Body',
       'Category One Defects', 'Category Two Defects', 'Certification Address',
       'Certification Body', 'Certification Contact', 'Clean Cup', 'Color',
       'Company', 'Country of Origin', 'Cupper Points',
       'Cupping Protocol and Descriptors', 'Expiration', 'Farm Name', 'Flavor',
       'Grading Date', 'Harvest Year', 'ICO Number', 'In-Country Partner',
       'Lot Number', 'Mill', 'Moisture', 'NA', 'NA.1', 'NA.2', 'NA.3',
       'Number of Bags', 'Owner', 'Owner.1', 'Processing Method', 'Producer',
       'Quakers', 'Region', 'Request a Sample', 'Species', 'Status',
       'Total Cup Points', 'Unnamed: 0', 'Variety',
       'View Green Analysis Details', 'quality_score', 'view_certificate_1',
       'view_certificate_2'],
      dtype='object')

Index(['Aftertaste', 'Altitude', 'Bag Weight', 'Balance', 'Body',
       'Category One Defects', 'Category Two Defects', 'Certification Address',
       'Certification Body', 'Certification Contact', 'Clean Cup', 'Color',
       'Company', 'Country of Origin', 'Cupper Points',
       'Cupping Protocol and Descriptors', 'Expiration', 'Farm Name', 'Flavor',
       'Grading Date', 'Harvest Year', 'ICO Number', 'In-Country Partner',
       'Lot Number', 'Mill', 'Moisture', 'NA', 'NA.1', 'NA.2', 'NA.3',
       'Number of Bags', 'Owner', 'Owner.1', 'Processing Method', 'Producer',
       'Quakers', 'Region', 'Request a Sample', 'Species', 'Status',
       'Total Cup Points', 'Unnamed: 0', 'Variety',
       'View Green Analysis Details', 'quality_score', 'view_certificate_1',
       'view_certificate_2'],
      dtype='object')

In [ ]:
#Cleaning

# Drop useless index-like column if present
if "Unnamed: 0" in coffee.columns:
    coffee = coffee.drop(columns=["Unnamed: 0"])

# Standardize text spacing a bit
for col in coffee.select_dtypes(include="object").columns:
    coffee[col] = coffee[col].astype(str).str.strip()

# Convert obvious numeric columns
numeric_cols_try = [
    "Aroma", "Flavor", "Aftertaste", "Acidity", "Body", "Balance",
    "Uniformity", "Clean.Cup", "Sweetness", "Cupper.Points",
    "Total.Cup.Points", "Moisture", "Category.One.Defects",
    "Category.Two.Defects", "Quakers", "Number.of.Bags",
    "altitude_low_meters", "altitude_high_meters", "altitude_mean_meters"
]

for col in numeric_cols_try:
    if col in coffee.columns:
        coffee[col] = pd.to_numeric(coffee[col], errors="coerce")

# Keep altitude_mean_meters as your main altitude field
if "altitude_mean_meters" not in coffee.columns and "Altitude" in coffee.columns:
    coffee["altitude_mean_meters"] = pd.to_numeric(
        coffee["Altitude"].astype(str).str.extract(r"(\d+\.?\d*)")[0],
        errors="coerce"
    )

print(coffee.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 47 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Aftertaste                        1340 non-null   float64
 1   Altitude                          1340 non-null   object 
 2   Bag Weight                        1340 non-null   object 
 3   Balance                           1340 non-null   float64
 4   Body                              1340 non-null   float64
 5   Category One Defects              1340 non-null   object 
 6   Category Two Defects              1340 non-null   object 
 7   Certification Address             1340 non-null   object 
 8   Certification Body                1340 non-null   object 
 9   Certification Contact             1340 non-null   object 
 10  Clean Cup                         1340 non-null   float64
 11  Color                             1340 non-null   object 
 12  Compan

In [ ]:
#missing value check
missing_summary = (
    coffee.isna().mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "pct_missing"]

display(missing_summary.head(20))

,column,pct_missing
0,view_certificate_2,1.000000
1,view_certificate_1,1.000000
2,View Green Analysis Details,1.000000
3,Moisture,1.000000
4,NA.2,1.000000
5,NA.3,1.000000
6,Cupping Protocol and Descriptors,1.000000
7,Request a Sample,1.000000
8,altitude_mean_meters,0.171642
9,Quakers,0.000746


### Statistical Analysis

In [ ]:
print('--- Quality Score Statistics ---')
print(coffee['quality_score'].describe())
print(f"Mode: {coffee['quality_score'].mode()[0]:.2f}")

print('\n--- Altitude Mean Meters Statistics ---')
print(coffee['altitude_mean_meters'].describe())
print(f"Mode: {coffee['altitude_mean_meters'].mode()[0]:.2f}")

print('\n--- Country of Origin Counts ---')
print(coffee['Country of Origin'].value_counts().head(5))

print('\n--- Processing Method Counts ---')
print(coffee['Processing Method'].value_counts().head(5))

print('\n--- Flavor Statistics ---')
print(coffee['Flavor'].describe())
print(f"Mode: {coffee['Flavor'].mode()[0]:.2f}")

--- Quality Score Statistics ---
count    1340.000000
mean       82.060776
std         3.657542
min         0.000000
25%        81.080000
50%        82.500000
75%        83.670000
max        90.580000
Name: quality_score, dtype: float64
Mode: 83.00

--- Altitude Mean Meters Statistics ---
count      1110.000000
mean       1682.541457
std        5763.290376
min           0.000000
25%        1040.000000
50%        1350.000000
75%        1650.000000
max      190164.000000
Name: altitude_mean_meters, dtype: float64
Mode: 1200.00

--- Country of Origin Counts ---
Country of Origin
Mexico       236
Colombia     183
Guatemala    181
Brazil       132
Taiwan        75
Name: count, dtype: int64

--- Processing Method Counts ---
Processing Method
Washed / Wet                 815
Natural / Dry                258
nan                          171
Semi-washed / Semi-pulped     56
Other                         26
Name: count, dtype: int64

--- Flavor Statistics ---
count    1340.000000
mean        7.5

### Data Visualization after Individual Attributes


*   Distribution of Coffee Quality Score
*   Distribution of Mean Altitude
*   Top 10 Countries of Origin
*   Distribution of Processing Method
*   Correlation Matrix of Sensory Attributes and Quality Score



In [ ]:
# Define a coffee-themed color palette for consistency
coffee_color_palette = ['#4a2c2a', '#a0522d', '#7b3f00', '#d2b48c', '#fffacd'] # Espresso, Roasted Coffee, Coffee Bean, Crema, Milk Foam

In [ ]:
fig = px.histogram(coffee, x='quality_score', nbins=20, title='Distribution of Coffee Quality Score', color_discrete_sequence=[coffee_color_palette[1]])
fig.update_layout(xaxis_title='Quality Score', yaxis_title='Count',
                  plot_bgcolor='#F6EBDD', paper_bgcolor='#F6EBDD') # Light beige background
fig.show()

In [ ]:
fig = px.histogram(coffee, x='altitude_mean_meters', nbins=30, title='Distribution of Mean Altitude', color_discrete_sequence=[coffee_color_palette[2]])
fig.update_layout(xaxis_title='Altitude (meters)', yaxis_title='Count',
                  plot_bgcolor='#F6EBDD', paper_bgcolor='#F6EBDD') # Light beige background
fig.show()

In [ ]:
fig = px.bar(coffee['Country of Origin'].value_counts().head(10).reset_index(),
             x='Country of Origin', y='count', title='Top 10 Countries of Origin',
             color='Country of Origin', color_discrete_sequence=coffee_color_palette)
fig.update_layout(xaxis_title='Country of Origin', yaxis_title='Count',
                  plot_bgcolor='#F6EBDD', paper_bgcolor='#F6EBDD') # Light beige background
fig.show()

In [ ]:
if 'Processing Method' in coffee.columns:
    filtered_processing_method = coffee[coffee['Processing Method'] != 'nan']['Processing Method']
    fig = px.bar(filtered_processing_method.value_counts().reset_index(),
                 x='Processing Method', y='count', title='Distribution of Processing Method',
                 color='Processing Method', color_discrete_sequence=coffee_color_palette)
    fig.update_layout(xaxis_title='Processing Method', yaxis_title='Count',
                      plot_bgcolor='#F6EBDD', paper_bgcolor='#F6EBDD') # Light beige background
    fig.show()

In [ ]:
#Define Sensory Columns
sensory_cols = [
    "Flavor", "Aftertaste", "Body", "Balance",
    "Clean Cup", "Cupper Points"
]
correlation_matrix = coffee[sensory_cols + ['quality_score']].corr().round(2)
fig = px.imshow(correlation_matrix,
                text_auto=True,
                title='Correlation Heatmap of Sensory Attributes and Quality Score',
                color_continuous_scale=[
                    '#4a2c2a',  # Dark Brown (espresso)
                    '#7b3f00',  # Medium Brown (coffee bean)
                    '#a0522d',  # Sienna (roasted coffee)
                    '#d2b48c',  # Tan (crema)
                    '#fffacd'   # Lemon Chiffon (milk foam)
                ])
fig.update_layout(plot_bgcolor='#F6EBDD', paper_bgcolor='#F6EBDD') # Light beige background
fig.show()

In [ ]:
fig = px.histogram(coffee, x='Flavor', nbins=20, title='Distribution of Flavor', color_discrete_sequence=[coffee_color_palette[0]])
fig.update_layout(xaxis_title='Flavor Score', yaxis_title='Count',
                  plot_bgcolor='#F6EBDD', paper_bgcolor='#F6EBDD') # Light beige background
fig.show()

### Pre-Processing for the Model

In [ ]:
#Already defined sensory columns above

#Define Target Variable
target_col = "quality_score"

#Choose predictors for quality model
candidate_features = [
    "Species",
    "Country.of.Origin",
    "Region",
    "Variety",
    "Processing.Method",
    "Color",
    "Moisture",
    "Category.One.Defects",
    "Category.Two.Defects",
    "Quakers",
    "Number.of.Bags",
    "altitude_mean_meters"
]

# Keep only columns that actually exist
quality_features = [c for c in candidate_features if c in coffee.columns]

# Build modeling dataframe
quality_df = coffee[quality_features + [target_col]].copy()

# Drop rows with missing target
quality_df = quality_df.dropna(subset=[target_col]).copy()

print("Quality model data shape:", quality_df.shape)
display(quality_df.head())

Quality model data shape: (1340, 8)


,Species,Region,Variety,Color,Moisture,Quakers,altitude_mean_meters,quality_score
0,Arabica,GUJI-HAMBELA/GOYO,nan,Green,NaN,0.0,1950.0,90.58
1,Arabica,GUJI-HAMBELA/ALAKA,Other,Green,NaN,0.0,1950.0,89.92
2,Arabica,nan,Bourbon,nan,NaN,0.0,1600.0,89.75
3,Arabica,Oromia,nan,Green,NaN,0.0,1800.0,89.00
4,Arabica,GUJI-HAMBELA/BISHAN FUGU,Other,Green,NaN,0.0,1950.0,88.83


In [ ]:
# EDA Process

# Data partitioning
train_df, temp_df = train_test_split(
    quality_df,
    test_size=0.30,
    random_state=30220
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=30220
)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

Train shape: (938, 8)
Validation shape: (201, 8)
Test shape: (201, 8)


In [ ]:
#Preprocessing pipeline
X_train = train_df[quality_features]
y_train = train_df[target_col]

X_valid = valid_df[quality_features]
y_valid = valid_df[target_col]

X_test = test_df[quality_features]
y_test = test_df[target_col]

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

### Prediction Models

Initial Two Models: Linear Regression and Random Forest

In [ ]:
#Model 1: Linear Regression

linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

valid_pred_lr = linear_model.predict(X_valid)
test_pred_lr = linear_model.predict(X_test)

lr_valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred_lr))
lr_valid_r2 = r2_score(y_valid, valid_pred_lr)

lr_test_rmse = np.sqrt(mean_squared_error(y_test, test_pred_lr))
lr_test_r2 = r2_score(y_test, test_pred_lr)

print("LINEAR REGRESSION")
print("Validation RMSE:", round(lr_valid_rmse, 3))
print("Validation R^2 :", round(lr_valid_r2, 3))
print("Test RMSE      :", round(lr_test_rmse, 3))
print("Test R^2       :", round(lr_test_r2, 3))

LINEAR REGRESSION
Validation RMSE: 3.167
Validation R^2 : -1.032
Test RMSE      : 3.142
Test R^2       : -0.357


In [ ]:
#Model 2: Random Forest

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=30220
    ))
])

rf_model.fit(X_train, y_train)

valid_pred_rf = rf_model.predict(X_valid)
test_pred_rf = rf_model.predict(X_test)

rf_valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred_rf))
rf_valid_r2 = r2_score(y_valid, valid_pred_rf)

rf_test_rmse = np.sqrt(mean_squared_error(y_test, test_pred_rf))
rf_test_r2 = r2_score(y_test, test_pred_rf)

print("RANDOM FOREST")
print("Validation RMSE:", round(rf_valid_rmse, 3))
print("Validation R^2 :", round(rf_valid_r2, 3))
print("Test RMSE      :", round(rf_test_rmse, 3))
print("Test R^2       :", round(rf_test_r2, 3))

RANDOM FOREST
Validation RMSE: 2.569
Validation R^2 : -0.337
Test RMSE      : 2.778
Test R^2       : -0.06


In [ ]:
#compare both methods

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "Validation_RMSE": [lr_valid_rmse, rf_valid_rmse],
    "Validation_R2": [lr_valid_r2, rf_valid_r2],
    "Test_RMSE": [lr_test_rmse, rf_test_rmse],
    "Test_R2": [lr_test_r2, rf_test_r2]
})

display(results.sort_values("Test_RMSE"))

,Model,Validation_RMSE,Validation_R2,Test_RMSE,Test_R2
1,Random Forest,2.569088,-0.336772,2.777981,-0.060437
0,Linear Regression,3.167240,-1.031709,3.142058,-0.356609


*Feature Importance*

In [ ]:
# Get trained random forest model and feature names
rf_fitted = rf_model.named_steps["model"]

# Get the preprocessor from the pipeline
preprocessor_fitted = rf_model.named_steps["preprocessor"]

# Get the feature names out from the preprocessor after transformation
# This will give the correct order and number of features corresponding to feature_importances_
all_feature_names = preprocessor_fitted.get_feature_names_out()

importances = rf_fitted.feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(15))

,feature,importance
1,num__altitude_mean_meters,0.369452
91,cat__Region_Comayagua,0.198656
411,cat__Variety_Typica,0.068304
391,cat__Variety_Caturra,0.056679
416,cat__Color_Green,0.047572
390,cat__Variety_Catuai,0.030014
160,cat__Region_KONA,0.023983
242,cat__Region_Oriente,0.023307
388,cat__Variety_Bourbon,0.021634
413,cat__Variety_nan,0.021431


Attempt to Lower RMSE:

In [ ]:
#Cap altitude outliers before modeling as anything abouve 4000m is certainly an entry error
coffee["altitude_clean"] = coffee["altitude_mean_meters"].clip(upper=4000)

#Consider sensory features to your quality model since these are the real drivers of quality_score
quality_features_v2 = [
    "Flavor", "Aftertaste", "Body", "Balance",
    "Clean Cup", "Cupper Points",          # sensory — high correlation
    "Species",                              # low cardinality categorical
    "Processing Method",                    # low cardinality categorical
    "altitude_clean",                       # cleaned numeric
    "Category One Defects",                 # numeric defect count
    "Category Two Defects",                 # numeric defect count
    "Quakers",                              # numeric
]
quality_features_v2 = [c for c in quality_features_v2 if c in coffee.columns]

#Rebuild modeling dataframe
quality_df_v2 = coffee[quality_features_v2 + ["quality_score"]].copy()
quality_df_v2 = quality_df_v2.dropna(subset=["quality_score"]).copy()

train_df2, temp_df2 = train_test_split(quality_df_v2, test_size=0.30, random_state=30220)
valid_df2, test_df2 = train_test_split(temp_df2, test_size=0.50, random_state=30220)

X_train2 = train_df2[quality_features_v2]
y_train2 = train_df2["quality_score"]

X_valid2 = valid_df2[quality_features_v2]
y_valid2 = valid_df2["quality_score"]

X_test2  = test_df2[quality_features_v2]
y_test2  = test_df2["quality_score"]

#Rebuild preprocessor
numeric_features2     = X_train2.select_dtypes(include=np.number).columns.tolist()
categorical_features2 = [c for c in X_train2.columns if c not in numeric_features2]

numeric_transformer2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])
categorical_transformer2 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor2 = ColumnTransformer([
    ("num", numeric_transformer2, numeric_features2),
    ("cat", categorical_transformer2, categorical_features2)
])
#Model 1 (v2): Linear Regression
linear_model2 = Pipeline([
    ("preprocessor", preprocessor2),
    ("model", LinearRegression())
])
linear_model2.fit(X_train2, y_train2)

valid_pred_lr2 = linear_model2.predict(X_valid2)
test_pred_lr2  = linear_model2.predict(X_test2)

print("LINEAR REGRESSION v2")
print("Validation RMSE:", round(np.sqrt(mean_squared_error(y_valid2, valid_pred_lr2)), 3))
print("Validation R²  :", round(r2_score(y_valid2, valid_pred_lr2), 3))
print("Test RMSE      :", round(np.sqrt(mean_squared_error(y_test2, test_pred_lr2)), 3))
print("Test R²        :", round(r2_score(y_test2, test_pred_lr2), 3))

#Model 2 (v2): Random Forest
rf_model2 = Pipeline([
    ("preprocessor", preprocessor2),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=8,           # reduced to prevent overfitting
        min_samples_split=15,  # increased for better generalization
        min_samples_leaf=8,    # increased for better generalization
        max_features=0.6,      # use subset of features per split
        random_state=30220
    ))
])
rf_model2.fit(X_train2, y_train2)

valid_pred_rf2 = rf_model2.predict(X_valid2)
test_pred_rf2  = rf_model2.predict(X_test2)

print("\nRANDOM FOREST v2")
print("Validation RMSE:", round(np.sqrt(mean_squared_error(y_valid2, valid_pred_rf2)), 3))
print("Validation R²  :", round(r2_score(y_valid2, valid_pred_rf2), 3))
print("Test RMSE      :", round(np.sqrt(mean_squared_error(y_test2, test_pred_rf2)), 3))
print("Test R²        :", round(r2_score(y_test2, test_pred_rf2), 3))

#Updated comparison table
results_v2 = pd.DataFrame({
    "Model": ["Linear Regression v2", "Random Forest v2"],
    "Validation_RMSE": [
        np.sqrt(mean_squared_error(y_valid2, valid_pred_lr2)),
        np.sqrt(mean_squared_error(y_valid2, valid_pred_rf2))
    ],
    "Validation_R2": [
        r2_score(y_valid2, valid_pred_lr2),
        r2_score(y_valid2, valid_pred_rf2)
    ],
    "Test_RMSE": [
        np.sqrt(mean_squared_error(y_test2, test_pred_lr2)),
        np.sqrt(mean_squared_error(y_test2, test_pred_rf2))
    ],
    "Test_R2": [
        r2_score(y_test2, test_pred_lr2),
        r2_score(y_test2, test_pred_rf2)
    ]
})
display(results_v2.sort_values("Test_RMSE"))

LINEAR REGRESSION v2
Validation RMSE: 1.056
Validation R²  : 0.774
Test RMSE      : 0.819
Test R²        : 0.908

RANDOM FOREST v2
Validation RMSE: 0.761
Validation R²  : 0.883
Test RMSE      : 1.143
Test R²        : 0.821


,Model,Validation_RMSE,Validation_R2,Test_RMSE,Test_R2
0,Linear Regression v2,1.056227,0.774049,0.818833,0.907867
1,Random Forest v2,0.761320,0.882609,1.142528,0.820626


### Tightening Random Forest and Introducing Gradient Boosting + Cross Validation

In [ ]:
# Tighter RF to reduce overfitting
rf_model3 = Pipeline([
    ("preprocessor", preprocessor2),
    ("model", RandomForestRegressor(
        n_estimators=500,
        max_depth=6,            # tighter than v2's 8
        min_samples_split=20,
        min_samples_leaf=10,
        max_features=0.5,
        random_state=30220
    ))
])
rf_model3.fit(X_train2, y_train2)

valid_pred_rf3 = rf_model3.predict(X_valid2)
test_pred_rf3  = rf_model3.predict(X_test2)

print("RANDOM FOREST v3 (tighter regularization)")
print("Validation RMSE:", round(np.sqrt(mean_squared_error(y_valid2, valid_pred_rf3)), 3))
print("Validation R²  :", round(r2_score(y_valid2, valid_pred_rf3), 3))
print("Test RMSE      :", round(np.sqrt(mean_squared_error(y_test2, test_pred_rf3)), 3))
print("Test R²        :", round(r2_score(y_test2, test_pred_rf3), 3))

RANDOM FOREST v3 (tighter regularization)
Validation RMSE: 0.734
Validation R²  : 0.891
Test RMSE      : 1.182
Test R²        : 0.808


In [ ]:
#Model 3: Gradient Boosting
gb_model = Pipeline([
    ("preprocessor", preprocessor2),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,     # slow learning = less overfit
        max_depth=4,            # shallow trees
        min_samples_split=15,
        min_samples_leaf=8,
        subsample=0.8,          # row sampling = less overfit
        random_state=30220
    ))
])
gb_model.fit(X_train2, y_train2)

valid_pred_gb = gb_model.predict(X_valid2)
test_pred_gb  = gb_model.predict(X_test2)

print("\nGRADIENT BOOSTING")
print("Validation RMSE:", round(np.sqrt(mean_squared_error(y_valid2, valid_pred_gb)), 3))
print("Validation R²  :", round(r2_score(y_valid2, valid_pred_gb), 3))
print("Test RMSE      :", round(np.sqrt(mean_squared_error(y_test2, test_pred_gb)), 3))
print("Test R²        :", round(r2_score(y_test2, test_pred_gb), 3))

#Cross-validation to get honest generalization estimate
# Combine train+valid for CV (don't use test)
X_trainval = pd.concat([X_train2, X_valid2])
y_trainval = pd.concat([y_train2, y_valid2])

for name, pipe in [("RF v3", rf_model3), ("Gradient Boosting", gb_model)]:
    # refit on combined train+valid for CV
    cv_pipe = Pipeline(pipe.steps)  # fresh copy
    scores = cross_val_score(
        cv_pipe, X_trainval, y_trainval,
        cv=5, scoring="neg_root_mean_squared_error"
    )


GRADIENT BOOSTING
Validation RMSE: 1.051
Validation R²  : 0.776
Test RMSE      : 1.017
Test R²        : 0.858


In [ ]:
    print(f"\n{name} — 5-fold CV RMSE: {-scores.mean():.3f} ± {scores.std():.3f}")


Gradient Boosting — 5-fold CV RMSE: 1.797 ± 1.430


In [ ]:
#Refined Gradient Boosting
gb_model2 = Pipeline([
    ("preprocessor", preprocessor2),
    ("model", GradientBoostingRegressor(
        n_estimators=500,       # more trees to compensate for slower learning
        learning_rate=0.03,     # slower than 0.05 = less overfit
        max_depth=3,            # shallower than before
        min_samples_split=20,
        min_samples_leaf=10,
        subsample=0.75,         # slightly more aggressive row sampling
        max_features=0.8,       # feature sampling per split
        random_state=30220
    ))
])
gb_model2.fit(X_train2, y_train2)

valid_pred_gb2 = gb_model2.predict(X_valid2)
test_pred_gb2  = gb_model2.predict(X_test2)

print("GRADIENT BOOSTING v2")
print("Validation RMSE:", round(np.sqrt(mean_squared_error(y_valid2, valid_pred_gb2)), 3))
print("Validation R²  :", round(r2_score(y_valid2, valid_pred_gb2), 3))
print("Test RMSE      :", round(np.sqrt(mean_squared_error(y_test2, test_pred_gb2)), 3))
print("Test R²        :", round(r2_score(y_test2, test_pred_gb2), 3))


GRADIENT BOOSTING v2
Validation RMSE: 0.819
Validation R²  : 0.864
Test RMSE      : 0.948
Test R²        : 0.877


### Final Comparison Table of all Prediction Models

In [ ]:
# ── Final comparison table ───────────────────────────────────────────
results_final = pd.DataFrame({
    "Model": [
        "Linear Regression", # v1
        "Random Forest",     # v1
        "Linear Regression v2",
        "Random Forest v2",
        "Random Forest v3",
        "Gradient Boosting",
        "Gradient Boosting v2"
    ],
    "Validation_RMSE": [
        lr_valid_rmse,
        rf_valid_rmse,
        np.sqrt(mean_squared_error(y_valid2, valid_pred_lr2)),
        np.sqrt(mean_squared_error(y_valid2, valid_pred_rf2)),
        np.sqrt(mean_squared_error(y_valid2, valid_pred_rf3)),
        np.sqrt(mean_squared_error(y_valid2, valid_pred_gb)),
        np.sqrt(mean_squared_error(y_valid2, valid_pred_gb2))
    ],
    "Validation_R2": [
        lr_valid_r2,
        rf_valid_r2,
        r2_score(y_valid2, valid_pred_lr2),
        r2_score(y_valid2, valid_pred_rf2),
        r2_score(y_valid2, valid_pred_rf3),
        r2_score(y_valid2, valid_pred_gb),
        r2_score(y_valid2, valid_pred_gb2)
    ],
    "Test_RMSE": [
        lr_test_rmse,
        rf_test_rmse,
        np.sqrt(mean_squared_error(y_test2, test_pred_lr2)),
        np.sqrt(mean_squared_error(y_test2, test_pred_rf2)),
        np.sqrt(mean_squared_error(y_test2, test_pred_rf3)),
        np.sqrt(mean_squared_error(y_test2, test_pred_gb)),
        np.sqrt(mean_squared_error(y_test2, test_pred_gb2))
    ],
    "Test_R2": [
        lr_test_r2,
        rf_test_r2,
        r2_score(y_test2, test_pred_lr2),
        r2_score(y_test2, test_pred_rf2),
        r2_score(y_test2, test_pred_rf3),
        r2_score(y_test2, test_pred_gb),
        r2_score(y_test2, test_pred_gb2)
    ]
})

# Calculate the Validation to Test RMSE Gap
results_final["Val_to_Test_RMSE_Gap"] = np.abs(results_final["Validation_RMSE"] - results_final["Test_RMSE"])

# Round numerical columns to two decimal points
for col in ["Validation_RMSE", "Validation_R2", "Test_RMSE", "Test_R2", "Val_to_Test_RMSE_Gap"]:
    results_final[col] = results_final[col].round(2)

display(results_final.sort_values("Test_RMSE"))

,Model,Validation_RMSE,Validation_R2,Test_RMSE,Test_R2,Val_to_Test_RMSE_Gap
2,Linear Regression v2,1.06,0.77,0.82,0.91,0.24
6,Gradient Boosting v2,0.82,0.86,0.95,0.88,0.13
5,Gradient Boosting,1.05,0.78,1.02,0.86,0.03
3,Random Forest v2,0.76,0.88,1.14,0.82,0.38
4,Random Forest v3,0.73,0.89,1.18,0.81,0.45
1,Random Forest,2.57,-0.34,2.78,-0.06,0.21
0,Linear Regression,3.17,-1.03,3.14,-0.36,0.03


### Visualizing the Models

In [ ]:
plot_data_lr = pd.DataFrame({
    'Actual': y_test,
    'Predicted': test_pred_lr,
    'Model': 'Linear Regression v1'
})

plot_data_rf = pd.DataFrame({
    'Actual': y_test,
    'Predicted': test_pred_rf,
    'Model': 'Random Forest v1'
})

plot_data_lr2 = pd.DataFrame({
    'Actual': y_test2,
    'Predicted': test_pred_lr2,
    'Model': 'Linear Regression v2'
})

plot_data_rf2 = pd.DataFrame({
    'Actual': y_test2,
    'Predicted': test_pred_rf2,
    'Model': 'Random Forest v2'
})

plot_data_rf3 = pd.DataFrame({
    'Actual': y_test2,
    'Predicted': test_pred_rf3,
    'Model': 'Random Forest v3'
})

plot_data_gb = pd.DataFrame({
    'Actual': y_test2,
    'Predicted': test_pred_gb,
    'Model': 'Gradient Boosting v1'
})

plot_data_gb2 = pd.DataFrame({
    'Actual': y_test2,
    'Predicted': test_pred_gb2,
    'Model': 'Gradient Boosting v2'
})

plot_data = pd.concat([
    plot_data_lr,
    plot_data_rf,
    plot_data_lr2,
    plot_data_rf2,
    plot_data_rf3,
    plot_data_gb,
    plot_data_gb2
])

In [ ]:
color_map = {
    'Linear Regression v1': 'rgb(173, 216, 230)',  # Light Blue
    'Linear Regression v2': 'rgb(30, 144, 255)',   # Dodger Blue
    'Random Forest v1': 'rgb(144, 238, 144)',      # Light Green
    'Random Forest v2': 'rgb(60, 179, 113)',       # Medium Sea Green
    'Random Forest v3': 'rgb(34, 139, 34)',        # Forest Green
    'Gradient Boosting v1': 'rgb(255, 165, 0)',    # Orange
    'Gradient Boosting v2': 'rgb(255, 69, 0)'      # Orange Red
}

# Find min/max values for the diagonal line from the combined plot_data
min_val = min(plot_data['Actual'].min(), plot_data['Predicted'].min())
max_val = max(plot_data['Actual'].max(), plot_data['Predicted'].max())

# Create the scatter plot
fig = px.scatter(plot_data, x='Actual', y='Predicted', color='Model',
                 title='Model Predictions vs. Actual Quality Score (Test Set)',
                 color_discrete_map=color_map,
                 hover_data={'Actual': ':.2f', 'Predicted': ':.2f', 'Model': True},
                 width=900, height=600)

# Add a diagonal line for perfect predictions
fig.add_shape(type='line', line=dict(dash='dash', color='gray', width=2),
              x0=min_val, y0=min_val, x1=max_val, y1=max_val)

# Update layout to match coffee theme
fig.update_layout(
    xaxis_title='Actual Quality Score',
    yaxis_title='Predicted Quality Score',
    plot_bgcolor='#F6EBDD',  # Light beige background
    paper_bgcolor='#F6EBDD',  # Light beige paper background
    hovermode='closest'
)

fig.show()

In [ ]:
# Define the color_map (already available in the kernel state)
color_map = {
    'Linear Regression v1': 'rgb(173, 216, 230)',  # Light Blue
    'Linear Regression v2': 'rgb(30, 144, 255)',   # Dodger Blue
    'Random Forest v1': 'rgb(144, 238, 144)',      # Light Green
    'Random Forest v2': 'rgb(60, 179, 113)',       # Medium Sea Green
    'Random Forest v3': 'rgb(34, 139, 34)',        # Forest Green
    'Gradient Boosting v1': 'rgb(255, 165, 0)',    # Orange
    'Gradient Boosting v2': 'rgb(255, 69, 0)'      # Orange Red
}

# Create DataFrames for residuals for each model
residuals_df_lr_v1 = pd.DataFrame({
    'Model': 'Linear Regression v1',
    'Predicted': test_pred_lr,
    'Residuals': y_test - test_pred_lr
})

residuals_df_rf_v1 = pd.DataFrame({
    'Model': 'Random Forest v1',
    'Predicted': test_pred_rf,
    'Residuals': y_test - test_pred_rf
})

residuals_df_lr_v2 = pd.DataFrame({
    'Model': 'Linear Regression v2',
    'Predicted': test_pred_lr2,
    'Residuals': y_test2 - test_pred_lr2
})

residuals_df_rf_v2 = pd.DataFrame({
    'Model': 'Random Forest v2',
    'Predicted': test_pred_rf2,
    'Residuals': y_test2 - test_pred_rf2
})

residuals_df_rf_v3 = pd.DataFrame({
    'Model': 'Random Forest v3',
    'Predicted': test_pred_rf3,
    'Residuals': y_test2 - test_pred_rf3
})

residuals_df_gb_v1 = pd.DataFrame({
    'Model': 'Gradient Boosting v1',
    'Predicted': test_pred_gb,
    'Residuals': y_test2 - test_pred_gb
})

residuals_df_gb_v2 = pd.DataFrame({
    'Model': 'Gradient Boosting v2',
    'Predicted': test_pred_gb2,
    'Residuals': y_test2 - test_pred_gb2
})

# Concatenate all residuals DataFrames
residuals_plot_data = pd.concat([
    residuals_df_lr_v1,
    residuals_df_rf_v1,
    residuals_df_lr_v2,
    residuals_df_rf_v2,
    residuals_df_rf_v3,
    residuals_df_gb_v1,
    residuals_df_gb_v2
])

# Create the scatter plot for residuals
fig_residuals = px.scatter(residuals_plot_data, x='Predicted', y='Residuals', color='Model',
                           title='Residuals vs. Predicted Quality Score (Test Set)',
                           color_discrete_map=color_map,
                           hover_data={'Predicted': ':.2f', 'Residuals': ':.2f', 'Model': True},
                           width=900, height=600)

# Add a horizontal line at y=0
fig_residuals.add_shape(type='line', line=dict(dash='dash', color='gray', width=2),
                        x0=residuals_plot_data['Predicted'].min(), y0=0,
                        x1=residuals_plot_data['Predicted'].max(), y1=0)

# Update layout to match coffee theme
fig_residuals.update_layout(
    xaxis_title='Predicted Quality Score',
    yaxis_title='Residuals (Actual - Predicted)',
    plot_bgcolor='#F6EBDD',  # Light beige background
    paper_bgcolor='#F6EBDD',  # Light beige paper background
    hovermode='closest'
)

fig_residuals.show()

### Building the Recommender System

In [ ]:
recommender_cols = [
    "Species",
    "Country of Origin",
    "Region",
    "Variety",
    "Processing Method",
    "altitude_mean_meters",
    "Aroma",
    "Flavor",
    "Aftertaste",
    "Acidity",
    "Body",
    "Balance",
    "Uniformity",
    "Clean Cup",
    "Sweetness",
    "Total Cup Points"
]

recommender_cols = [c for c in recommender_cols if c in coffee.columns]

rec_df = coffee[recommender_cols].copy()

# Need the main preference variables present
preference_cols = [c for c in ["Flavor", "Acidity", "Sweetness", "Body", "Balance"] if c in rec_df.columns]

rec_df = rec_df.dropna(subset=preference_cols).reset_index(drop=True)

print("Recommendation dataset shape:", rec_df.shape)
display(rec_df.head())

Recommendation dataset shape: (1340, 12)


,Species,Country of Origin,Region,Variety,Processing Method,altitude_mean_meters,Flavor,Aftertaste,Body,Balance,Clean Cup,Total Cup Points
0,Arabica,Ethiopia,GUJI-HAMBELA/GOYO,nan,Washed / Wet,1950.0,8.83,8.67,8.50,8.42,10.0,Sample 90.58
1,Arabica,Ethiopia,GUJI-HAMBELA/ALAKA,Other,Washed / Wet,1950.0,8.67,8.50,8.42,8.42,10.0,Sample 89.92
2,Arabica,Guatemala,nan,Bourbon,nan,1600.0,8.50,8.42,8.33,8.42,10.0,Sample 89.75
3,Arabica,Ethiopia,Oromia,nan,Natural / Dry,1800.0,8.58,8.42,8.50,8.25,10.0,Sample 89.00
4,Arabica,Ethiopia,GUJI-HAMBELA/BISHAN FUGU,Other,Washed / Wet,1950.0,8.50,8.25,8.42,8.33,10.0,Sample 88.83


In [ ]:
# Extra Credit Model: KNN

# Scale preference dimensions for similarity search
pref_scaler = StandardScaler()
rec_features_scaled = pref_scaler.fit_transform(rec_df[preference_cols])

knn_model = NearestNeighbors(n_neighbors=5, metric="euclidean")
display(knn_model.fit(rec_features_scaled))

#add predicted quality to recommendation table
best_quality_model = rf_model  # switch to linear_model if that performs better

# Build a predicted quality column for rows where predictors exist
pred_input = rec_df.copy()

for col in quality_features:
    if col not in pred_input.columns:
        pred_input[col] = np.nan

pred_input = pred_input[quality_features]

rec_df["Predicted_Quality"] = best_quality_model.predict(pred_input)

NearestNeighbors(metric='euclidean')

In [ ]:
#Recommendation function
def recommend_coffees(country, species, processing_method, flavor, acidity, sweetness, body, balance):
    df = rec_df.copy()

    # Optional filters
    if country != "Any":
        df = df[df["Country of Origin"] == country]
    if species != "Any":
        df = df[df["Species"] == species]
    if "Processing Method" in df.columns and processing_method != "Any":
        df = df[df["Processing Method"] == processing_method]

    if len(df) < 5:
        return pd.DataFrame({"message": ["Not enough coffees match these filters. Try 'Any'."]})

    # Scale filtered set
    filtered_scaled = pref_scaler.transform(df[preference_cols])

    user_vector = np.array([[flavor, acidity, sweetness, body, balance]])
    user_scaled = pref_scaler.transform(user_vector)

    # Distance from user preference
    distances = np.linalg.norm(filtered_scaled - user_scaled, axis=1)

    df = df.copy()
    df["distance"] = distances

    # Higher quality + closer distance = better recommendation
    # Normalize both pieces
    df["distance_score"] = 1 / (1 + df["distance"])

    pq_min = df["Predicted_Quality"].min()
    pq_max = df["Predicted_Quality"].max()

    if pq_max > pq_min:
        df["quality_score_norm"] = (df["Predicted_Quality"] - pq_min) / (pq_max - pq_min)
    else:
        df["quality_score_norm"] = 0.5

    # Final weighted score
    df["recommendation_score"] = 0.7 * df["distance_score"] + 0.3 * df["quality_score_norm"]

    top5 = df.sort_values("recommendation_score", ascending=False).head(5).copy()

    display_cols = [
        c for c in [
            "Species", "Country of Origin", "Region", "Variety", "Processing Method",
            "Flavor", "Acidity", "Sweetness", "Body", "Balance",
            "Predicted_Quality", "Total Cup Points", "recommendation_score"
        ] if c in top5.columns
    ]

    return top5[display_cols].reset_index(drop=True)

#Plot function for the top 5
def plot_top5(country, species, processing_method, flavor, acidity, sweetness, body, balance):
    top5 = recommend_coffees(country, species, processing_method, flavor, acidity, sweetness, body, balance)

    if "message" in top5.columns:
        return px.scatter(title="No matching coffees found")

    fig = px.scatter(
        top5,
        x="Acidity",
        y="Sweetness",
        size="Predicted_Quality",
        hover_data=[c for c in ["Country.of.Origin", "Variety", "Flavor", "Body", "Balance"] if c in top5.columns],
        text=top5.index.astype(str),
        title="Top 5 Recommended Coffees: Acidity vs Sweetness"
    )

    fig.update_traces(textposition="top center")
    fig.update_layout(height=500)

    return fig

#dropdown choices
country_choices = ["Any"] + sorted([x for x in rec_df["Country of Origin"].dropna().unique().tolist() if x != "nan"])
species_choices = ["Any"] + sorted([x for x in rec_df["Species"].dropna().unique().tolist() if x != "nan"])

if "Processing Method" in rec_df.columns:
    processing_choices = ["Any"] + sorted([x for x in rec_df["Processing Method"].dropna().unique().tolist() if x != "nan"])
else:
    processing_choices = ["Any"]

###Building PerfectPour in Gradio

In [ ]:
# Updated Gradio app customization - PerfectPour
import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# =========================================================
# COFFEE DRINK CHOICES
# =========================================================
coffee_drink_choices = [
    "Iced latte",
    "Cold brew",
    "Flavored latte",
    "Cappuccino",
    "Flat white",
    "Americano (iced)",
    "Americano (hot)",
    "Mocha (iced)",
    "Mocha (hot)",
    "Macchiato (iced)",
    "Macchiato (hot)",
    "Black coffee (iced)",
    "Black coffee (hot)",
    "Shaken espresso",
    "Cortado"
]

# =========================================================
# MAP USER'S CAFE DRINK TO A SIMPLE TASTE PROFILE
# =========================================================
def coffee_choice_profile(drink):
    profiles = {
        "Iced latte": {"flavor": 7.0, "acidity": 4.5, "sweetness": 6.5},
        "Cold brew": {"flavor": 6.5, "acidity": 3.5, "sweetness": 4.5},
        "Flavored latte": {"flavor": 7.5, "acidity": 4.0, "sweetness": 8.0},
        "Cappuccino": {"flavor": 7.0, "acidity": 5.0, "sweetness": 5.0},
        "Flat white": {"flavor": 7.5, "acidity": 4.5, "sweetness": 5.0},
        "Americano (iced)": {"flavor": 6.0, "acidity": 5.5, "sweetness": 3.5},
        "Americano (hot)": {"flavor": 6.5, "acidity": 5.5, "sweetness": 3.0},
        "Mocha (iced)": {"flavor": 8.0, "acidity": 4.0, "sweetness": 8.0},
        "Mocha (hot)": {"flavor": 8.0, "acidity": 4.0, "sweetness": 7.5},
        "Macchiato (iced)": {"flavor": 7.5, "acidity": 5.0, "sweetness": 4.5},
        "Macchiato (hot)": {"flavor": 7.5, "acidity": 5.0, "sweetness": 4.0},
        "Black coffee (iced)": {"flavor": 6.5, "acidity": 6.0, "sweetness": 2.5},
        "Black coffee (hot)": {"flavor": 7.0, "acidity": 6.0, "sweetness": 2.0},
        "Shaken espresso": {"flavor": 8.0, "acidity": 5.5, "sweetness": 5.0},
        "Cortado": {"flavor": 7.5, "acidity": 4.5, "sweetness": 4.0}
    }
    return profiles.get(drink, {"flavor": 7.0, "acidity": 5.0, "sweetness": 5.0})


# =========================================================
# CREATE DRINK PROFILE DATAFRAME FOR PLOT
# =========================================================
drink_profile_df = pd.DataFrame([
    {
        "Coffee Type": drink,
        "Flavor": coffee_choice_profile(drink)["flavor"],
        "Acidity": coffee_choice_profile(drink)["acidity"],
        "Sweetness": coffee_choice_profile(drink)["sweetness"]
    }
    for drink in coffee_drink_choices
])


# =========================================================
# BLEND USER SLIDERS + CAFE DRINK STYLE
# =========================================================
def blended_preferences(flavor, acidity, sweetness, drink):
    drink_profile = coffee_choice_profile(drink)

    final_flavor = 0.75 * flavor + 0.25 * drink_profile["flavor"]
    final_acidity = 0.75 * acidity + 0.25 * drink_profile["acidity"]
    final_sweetness = 0.75 * sweetness + 0.25 * drink_profile["sweetness"]

    return final_flavor, final_acidity, final_sweetness


# =========================================================
# TAB 2 LIVE SUMMARY
# =========================================================
def live_preference_summary(flavor, acidity, sweetness, drink):
    final_flavor, final_acidity, final_sweetness = blended_preferences(
        flavor, acidity, sweetness, drink
    )

    return f"""
### Your current coffee profile
- **Taste/Aroma:** {final_flavor:.1f}
- **Acidity:** {final_acidity:.1f}
- **Sweetness:** {final_sweetness:.1f}
- **Typical coffee choice:** {drink}
"""


# =========================================================
# TAB 3 PLOT: WHERE YOU FALL
# =========================================================
def where_you_fall_plot(flavor, acidity, sweetness, drink):
    final_flavor, final_acidity, final_sweetness = blended_preferences(
        flavor, acidity, sweetness, drink
    )

    plot_df = drink_profile_df.copy()

    fig = go.Figure()

    # Other coffee drink types
    fig.add_trace(
        go.Scatter(
            x=plot_df["Acidity"],
            y=plot_df["Sweetness"],
            mode="markers",
            marker=dict(
                size=16,
                color="#7B3F00",  # chestnut brown
                opacity=0.85,
                line=dict(width=1.5, color="#5C3A21")
            ),
            customdata=np.stack(
                [
                    plot_df["Coffee Type"],
                    plot_df["Flavor"]
                ],
                axis=-1
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Flavor/Taste: %{customdata[1]:.1f}<br>"
                "Acidity: %{x:.1f}<br>"
                "Sweetness: %{y:.1f}<extra></extra>"
            ),
            showlegend=False
        )
    )

    # User profile point
    fig.add_trace(
        go.Scatter(
            x=[final_acidity],
            y=[final_sweetness],
            mode="markers",
            marker=dict(
                size=28,
                color="#C49A6C",  # lighter brown
                symbol="star",
                line=dict(width=2, color="#5C3A21")
            ),
            hovertemplate=(
                "<b>YOU</b><br>"
                f"Typical Order: {drink}<br>"
                f"Flavor/Taste: {final_flavor:.1f}<br>"
                f"Acidity: {final_acidity:.1f}<br>"
                f"Sweetness: {final_sweetness:.1f}<extra></extra>"
            ),
            showlegend=False
        )
    )

    fig.update_layout(
        height=600,
        plot_bgcolor="#F6EBDD",
        paper_bgcolor="#F6EBDD",
        font=dict(color="#7B3F00", family="Comic Sans MS"),
        title=dict(
            text="Where You Fall: Acidity vs. Sweetness",
            font=dict(size=26)
        ),
        xaxis=dict(
            title="Acidity Level",
            range=[1, 10],
            gridcolor="#D9C8A9"
        ),
        yaxis=dict(
            title="Sweetness Level",
            range=[1, 10],
            gridcolor="#D9C8A9"
        ),
        showlegend=False
    )

    return fig


# =========================================================
# TAB 4 FINAL RECOMMENDATION
# =========================================================
def recommend_drink_style(flavor, acidity, sweetness, drink):
    final_flavor, final_acidity, final_sweetness = blended_preferences(
        flavor, acidity, sweetness, drink
    )

    if final_sweetness >= 7.0 and final_flavor >= 7.0:
        rec_style = "Flavored latte or mocha"
        reason = "You seem to enjoy sweeter, richer coffee drinks with bold flavor."
    elif final_acidity <= 4.5 and final_sweetness <= 5.0:
        rec_style = "Cold brew or smooth iced latte"
        reason = "You lean toward smoother, less acidic coffees."
    elif final_acidity >= 6.0 and final_flavor >= 6.5:
        rec_style = "Black coffee or americano"
        reason = "Your profile suggests you enjoy brighter, sharper coffee characteristics."
    elif 4.5 <= final_sweetness <= 6.5 and 6.5 <= final_flavor:
        rec_style = "Cappuccino or flat white"
        reason = "You seem to prefer balanced coffees with a creamy but not overly sweet profile."
    else:
        rec_style = "Macchiato or cortado"
        reason = "Your preferences suggest you like a more focused espresso-forward drink with moderate sweetness."

    summary = f"""
## ☕ Our Recommendation for You

### Your curated flavor profile
- **Overall Flavor Preference (Taste/Aroma):** {final_flavor:.1f}
- **Acidity Level:** {final_acidity:.1f}
- **Sweetness Level:** {final_sweetness:.1f}
- **Typical Coffee Order:** {drink}

### Recommended drink style
**{rec_style}**

### Why we picked it
{reason}

This recommendation is based on your slider preferences plus the style of coffee you usually order at cafés.
"""

    top_matches = pd.DataFrame({
        "Category": ["Flavor", "Acidity", "Sweetness", "Typical Order", "Recommended Style"],
        "Your Profile": [
            round(final_flavor, 1),
            round(final_acidity, 1),
            round(final_sweetness, 1),
            drink,
            rec_style
        ]
    })

    return summary, top_matches


# =========================================================
# CUTE COFFEE THEME CSS
# =========================================================
custom_css = """
body {
    background: #EADCCB !important;
    color: #7B3F00 !important;
}

.gradio-container {
    font-family: 'Comic Sans MS', 'Comic Sans', cursive !important;
    background: #EADCCB !important;
    color: #7B3F00 !important;
}

.block, .gr-box, .gr-form, .gr-panel {
    background: #F6EBDD !important;
    border: 2px solid #5C3A21 !important;
    border-radius: 18px !important;
}

button[role="tab"] {
    background: #D9C8A9 !important;
    color: #7B3F00 !important;
    border: 2px solid #5C3A21 !important;
    border-bottom: none !important;
    font-family: 'Comic Sans MS', 'Comic Sans', cursive !important;
    font-size: 20px !important;
}

button[role="tab"][aria-selected="true"] {
    background: #F6EBDD !important;
    font-weight: bold !important;
}

label, p, h1, h2, h3, h4, span, div {
    color: #7B3F00 !important;
    font-family: 'Comic Sans MS', 'Comic Sans', cursive !important;
}

button {
    border-radius: 14px !important;
    border: 2px solid #5C3A21 !important;
    background: #E7D6B8 !important;
    color: #7B3F00 !important;
    font-family: 'Comic Sans MS', 'Comic Sans', cursive !important;
}

input, textarea, select {
    border: 2px solid #7A5230 !important;
    border-radius: 12px !important;
    background: #FFF9F1 !important;
    color: #7B3F00 !important;
    font-family: 'Comic Sans MS', 'Comic Sans', cursive !important;
}

/* Brown sliders */
input[type="range"] {
    accent-color: #5C3A21 !important;
}

.coffee-pattern {
    background:
        radial-gradient(circle at 20px 20px, rgba(183,123,67,0.30) 1.4px, transparent 1.5px),
        #F6EBDD !important;
    background-size: 22px 22px !important;
    border: 2px solid #5C3A21 !important;
    border-radius: 18px !important;
    padding: 18px !important;
}

.hero-box {
    background: #F6EBDD !important;
    border: 2px solid #5C3A21 !important;
    border-radius: 22px !important;
    padding: 26px !important;
    text-align: center !important;
}

.app-title {
    font-size: 42px !important;
    font-weight: 800 !important;
    margin-bottom: 8px !important;
    color: #7B3F00 !important;
}

.app-subtitle {
    font-size: 22px !important;
    font-style: italic !important;
    margin-bottom: 12px !important;
    color: #7B3F00 !important;
}

.created-by {
    font-size: 20px !important;
    margin-top: 14px !important;
    color: #7B3F00 !important;
}
"""


# =========================================================
# APP LAYOUT
# =========================================================
with gr.Blocks(css=custom_css, title="PerfectPour ☕") as app:

    with gr.Tabs():

        # -------------------------------------------------
        # TAB 1
        # -------------------------------------------------
        with gr.Tab("Start Here"):
            gr.HTML("""
            <div class="hero-box">
                <div class="app-title">☕ PerfectPour ☕</div>
                <div class="app-subtitle">Curating your perfect coffee match based on your taste preferences and café orders.</div>
                <div class="created-by"><b>Created By:</b><br>Reagan McGowan &amp; Ashly Turcios</div>
            </div>
            """)

            gr.Markdown("""
### Welcome to our coffee recommendation app

Our app is designed to personalize coffee suggestions based on the flavor characteristics you enjoy most.

You’ll be able to:
- choose your preferred **taste/aroma level**
- choose your preferred **acidity**
- choose your preferred **sweetness**
- select your **typical coffee order**
- see **where your taste profile falls**
- receive a **recommended drink style**
""")

        # -------------------------------------------------
        # TAB 2
        # -------------------------------------------------
        with gr.Tab("Your Preferences"):
            with gr.Column(elem_classes="coffee-pattern"):
                gr.Markdown("## Tell us what kind of coffee you love")

                flavor_in = gr.Slider(
                    1, 10, value=6.5, step=0.1,
                    label="Overall Flavor Preference: Taste/Aroma"
                )

                acidity_in = gr.Slider(
                    1, 10, value=5.5, step=0.1,
                    label="Acidity Level"
                )

                sweetness_in = gr.Slider(
                    1, 10, value=6.0, step=0.1,
                    label="Sweetness Level"
                )

                drink_in = gr.Dropdown(
                    choices=coffee_drink_choices,
                    value="Iced latte",
                    label="Your typical coffee of choice"
                )

                profile_preview = gr.Markdown()
                preview_btn = gr.Button("Save My Preferences")

                preview_btn.click(
                    fn=live_preference_summary,
                    inputs=[flavor_in, acidity_in, sweetness_in, drink_in],
                    outputs=profile_preview
                )

        # -------------------------------------------------
        # TAB 3
        # -------------------------------------------------
        with gr.Tab("Where You Fall"):
            with gr.Column(elem_classes="coffee-pattern"):
                gr.Markdown("## Where You Fall")
                gr.Markdown("*Hover over each dot to compare your profile with different coffee drink types.*")

                profile_btn = gr.Button("Show My Taste Profile")
                profile_plot = gr.Plot(label="Acidity vs Sweetness Coffee Map")

                profile_btn.click(
                    fn=where_you_fall_plot,
                    inputs=[flavor_in, acidity_in, sweetness_in, drink_in],
                    outputs=profile_plot
                )

        # -------------------------------------------------
        # TAB 4
        # -------------------------------------------------
        with gr.Tab("Your Perfect Coffee"):
            with gr.Column(elem_classes="coffee-pattern"):
                gr.Markdown("## Your curated coffee recommendation")

                rec_btn = gr.Button("Find My Perfect Coffee")
                rec_summary = gr.Markdown()
                rec_table = gr.Dataframe(
                    interactive=False,
                    wrap=True,
                    label="Your coffee profile summary"
                )

                rec_btn.click(
                    fn=recommend_drink_style,
                    inputs=[flavor_in, acidity_in, sweetness_in, drink_in],
                    outputs=[rec_summary, rec_table]
                )

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fabb44dc9f4282a8df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fabb44dc9f4282a8df.gradio.live
